In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

In [2]:
def code_choice(row):
    row = str(row).lower()
    if 'not' in row: 
        return 0
    elif 'drivable' in row:
        return 0
    else: 
        return 1

In [3]:
def process_baseline(filepath, name, resample_to_balanced=False):
    df = pd.read_csv(filepath)
    df['gt'] = df['choice'].apply(code_choice)
    
    # Handle different column names for predictions
    if 'nli_is_flooded' in df.columns:
        df['pred'] = df['nli_is_flooded'].astype(int)
    elif 'nli_label' in df.columns:
        # For Liu FloodVision, the positive label (flooded/actionable) is 'no, not passable'
        # All other labels ('uncertain', 'yes, passable') are considered negative
        df['pred'] = df['nli_label'].apply(lambda x: 1 if 'no, not passable' in str(x).lower() else 0)
    elif 'sentiment_1' in df.columns:
        # For inspection set, we have two questions (sentiment_1 and sentiment_2)
        if 'Basic' in name:
            df['pred'] = df['sentiment_2'].astype(int)
        else:
            df['pred'] = df['sentiment_1'].astype(int)
            
    if resample_to_balanced:
        # Resample to 250 images per predicted class (pred=1 and pred=0)
        df_pos = df[df['pred'] == 1].sample(n=250, random_state=42)
        df_neg = df[df['pred'] == 0].sample(n=250, random_state=42)
        df = pd.concat([df_pos, df_neg]).reset_index(drop=True)
    
    df['correct'] = df['pred'] == df['gt']
    
    print(f"--- {name} ---")
    print(f"GT Mean: {df['gt'].mean():.3f}")
    print(f"Correct Mean: {df['correct'].mean():.3f}")
    
    df_1 = df[df['pred'] == 1]
    df_0 = df[df['pred'] == 0]
    print(f"Counts: Pos={len(df_1)}, Neg={len(df_0)}")
    print(f"P(correct|pred=1): {df_1['correct'].mean():.3f}")
    print(f"P(correct|pred=0): {df_0['correct'].mean():.3f}")
    print()
    
    return df, df_1, df_0

In [ ]:
lyu_basic, lyu_basic_1, lyu_basic_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/lyu_basic_annotated.csv", "Lyu Basic")
lyu_adv, lyu_adv_1, lyu_adv_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/lyu_advanced_annotated.csv", "Lyu Advanced")
yang_adv, yang_adv_1, yang_adv_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/yang_advanced_annotated.csv", "Yang Advanced")
liu_fv, liu_fv_1, liu_fv_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/liu_floodvision_annotated.csv", "Liu FloodVision")
insp_basic, insp_basic_1, insp_basic_0 = process_baseline("../../data/processed/inspection_set.csv", "Ours (Basic Flooding Query)", resample_to_balanced=True)
insp_1ft, insp_1ft_1, insp_1ft_0 = process_baseline("../../data/processed/inspection_set.csv", "Ours (>1ft flooding)", resample_to_balanced=False)

In [5]:
def compare_baselines(name1, df1_1, df1_0, name2, df2_1, df2_0):
    print(f"=== T-Test Comparison: {name1} vs {name2} ===\n")

    # Predicted Positives Comparison
    t_pos, p_pos = ttest_ind(df1_1['correct'], df2_1['correct'])
    print(f"Predicted Positives (pred=1):")
    print(f"  t = {t_pos:.4f}, p = {p_pos:.4e}")
    print(f"  {name1} p(correct) = {df1_1['correct'].mean():.3f}")
    print(f"  {name2} p(correct) = {df2_1['correct'].mean():.3f}")
    print()

    # Predicted Negatives Comparison
    t_neg, p_neg = ttest_ind(df1_0['correct'], df2_0['correct'])
    print(f"Predicted Negatives (pred=0):")
    print(f"  t = {t_neg:.4f}, p = {p_neg:.4e}")
    print(f"  {name1} p(correct) = {df1_0['correct'].mean():.3f}")
    print(f"  {name2} p(correct) = {df2_0['correct'].mean():.3f}")
    print("\n")

compare_baselines("Lyu Basic", lyu_basic_1, lyu_basic_0, "Yang Advanced", yang_adv_1, yang_adv_0)
compare_baselines("Lyu Advanced", lyu_adv_1, lyu_adv_0, "Yang Advanced", yang_adv_1, yang_adv_0)
compare_baselines("Liu FloodVision", liu_fv_1, liu_fv_0, "Yang Advanced", yang_adv_1, yang_adv_0)
compare_baselines("Ours (Basic Flooding Query)", insp_basic_1, insp_basic_0, "Yang Advanced", yang_adv_1, yang_adv_0)
compare_baselines("Ours (>1ft flooding)", insp_1ft_1, insp_1ft_0, "Yang Advanced", yang_adv_1, yang_adv_0)

In [6]:
def get_metrics(df, name):
    tp = len(df[(df['pred'] == 1) & (df['gt'] == 1)])
    fp = len(df[(df['pred'] == 1) & (df['gt'] == 0)])
    fn = len(df[(df['pred'] == 0) & (df['gt'] == 1)])
    tn = len(df[(df['pred'] == 0) & (df['gt'] == 0)])
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    for_rate = fn / (fn + tn) if (fn + tn) > 0 else 0
    
    return {
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'CSI (Threat score)': csi,
        'FOR': for_rate
    }

results = [
    get_metrics(lyu_basic, "Lyu Basic"),
    get_metrics(lyu_adv, "Lyu Advanced"),
    get_metrics(liu_fv, "Liu FloodVision"),
    get_metrics(insp_basic, "Ours (Basic Flooding Query)"),
    get_metrics(insp_1ft, "Ours (>1ft flooding)"),
    get_metrics(yang_adv, "Yang Advanced (Ours)")
]

res_df = pd.DataFrame(results)
# Sort by Precision (which was P(correct|pred=1))
res_df = res_df.sort_values('Precision', ascending=False).reset_index(drop=True)

metrics = ['Precision', 'Recall', 'F1-score', 'CSI (Threat score)', 'FOR']
models = res_df['Model'].tolist()

latex_table = """\\begin{table}[ht]
\\small
\\centering
\\caption{Performance metrics across prompt-based baselines.}
\\label{tab:baseline_performance}
\\begin{tabular}{l" + "c" * len(models) + "}
\\toprule
Metric & " + " & ".join(models) + " \\\\
\\midrule
"""

for metric in metrics:
    row_vals = []
    for model in models:
        val = res_df[res_df['Model'] == model][metric].values[0]
        if metric == 'FOR':
            if val == 0:
                row_vals.append("0.000")
            elif val < 0.001:
                # Format in scientific notation like the example
                exponent = int(np.floor(np.log10(val)))
                mantissa = val / (10**exponent)
                row_vals.append(f"({mantissa:.2f} \\pm 0.00) \\cdot 10^{{{exponent}}}")
            else:
                row_vals.append(f"{val:.3f}")
        else:
            row_vals.append(f"{val:.3f}")
    latex_table += f"{metric} & " + " & ".join(row_vals) + " \\\\\\\\ \n"

latex_table += """\\bottomrule
\\end{tabular}
\\end{table}"""

print(latex_table)